# 📓 Notebook 1 — Data Cleaning & Preprocessing
**Hotel Dynamic Pricing Optimization | Big Data Analytics (404)**

This notebook covers the full data preprocessing pipeline:
1. Load & inspect the raw hotel booking dataset
2. Inject and handle missing values
3. Remove duplicates
4. Data type formatting
5. Outlier detection & treatment (IQR + Z-score)
6. Feature engineering (12 new features)
7. Categorical encoding (LabelEncoder)
8. Feature scaling (StandardScaler)


In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import LabelEncoder, StandardScaler
import pickle, os, warnings
warnings.filterwarnings("ignore")

%matplotlib inline
plt.rcParams.update({"figure.figsize": (12, 5), "axes.titlesize": 13})

RAW_PATH   = "../data/hotel_booking_data.csv"
CLEAN_PATH = "../data/hotel_booking_clean.csv"
MODEL_DIR  = "../outputs/models"
os.makedirs(MODEL_DIR, exist_ok=True)
print("✓ Libraries loaded")


## Step 1 — Load & Inspect Raw Dataset

In [ ]:
df = pd.read_csv(RAW_PATH, parse_dates=["checkin_date"])
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Memory: {df.memory_usage(deep=True).sum()/1e6:.2f} MB\n")
print("Column dtypes:")
print(df.dtypes.value_counts())
print(f"\nMissing values: {df.isnull().sum().sum()}")
print(f"Duplicate rows: {df.duplicated().sum()}")
df.head(3)


In [ ]:
# Key statistics on pricing columns
price_cols = ["base_price", "competitor_avg_price", "actual_price", "optimal_price"]
df[price_cols].describe().round(2)


## Step 2 — Inject Realistic Data Quality Issues
*(Required by rubric to demonstrate preprocessing capability)*

In [ ]:
rng = np.random.default_rng(99)
n = len(df)

# 1% missing values in 5 columns
for col in ["occupancy_rate", "reviews_score", "weather_score",
            "lead_time_days", "competitor_price_2"]:
    idx = rng.choice(n, size=int(n * 0.01), replace=False)
    df.loc[idx, col] = np.nan

# 0.3% duplicate rows
dup_idx = rng.choice(n, size=int(n * 0.003), replace=False)
df = pd.concat([df, df.iloc[dup_idx]], ignore_index=True)

# 15 extreme outliers in actual_price
out_idx = rng.choice(n, size=15, replace=False)
df.loc[out_idx, "actual_price"] *= rng.uniform(3, 6, size=15)

print(f"After injection: {df.shape[0]:,} rows")
print(f"Missing values:  {df.isnull().sum().sum()}")
print(f"Duplicates:      {df.duplicated().sum()}")
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
print("\nMissing % by column:")
print(missing_pct[missing_pct > 0])


## Step 3 — Handle Missing Values (Median/Mode Imputation)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Before
missing = df.isnull().sum()
missing = missing[missing > 0]
axes[0].bar(missing.index, missing.values, color="#e94560")
axes[0].set_title("Missing Values — Before Imputation")
axes[0].set_xticklabels(missing.index, rotation=30, ha="right")
axes[0].set_ylabel("Count")

# Impute
numeric_cols = df.select_dtypes(include=[np.number]).columns
cat_cols     = df.select_dtypes(include=["object"]).columns
for col in numeric_cols:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].median())
for col in cat_cols:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].mode()[0])

# After
missing_after = df.isnull().sum()
missing_after = missing_after[missing_after > 0]
if len(missing_after) == 0:
    axes[1].text(0.5, 0.5, "✓ No Missing Values", ha="center", va="center",
                 fontsize=14, color="green", transform=axes[1].transAxes)
    axes[1].set_title("Missing Values — After Imputation")
axes[1].axis("off")

plt.suptitle("Missing Value Treatment", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()
print(f"✓ Missing values after imputation: {df.isnull().sum().sum()}")


## Step 4 — Remove Duplicate Rows

In [ ]:
before = len(df)
df = df.drop_duplicates(subset=[c for c in df.columns if c != "booking_id"])
df = df.reset_index(drop=True)
print(f"Rows before: {before:,} | Rows after: {len(df):,}")
print(f"Removed {before - len(df)} duplicates")


## Step 5 — Data Type Conversion

In [ ]:
# Ensure correct types
if not pd.api.types.is_datetime64_any_dtype(df["checkin_date"]):
    df["checkin_date"] = pd.to_datetime(df["checkin_date"])

int_cols   = ["year","month","day_of_week","is_weekend","is_holiday",
              "lead_time_days","length_of_stay","event_magnitude",
              "weather_score","repeat_guest","cancellation"]
float_cols = ["base_price","occupancy_rate","demand_score",
              "optimal_price","actual_price","revenue_per_night","revpar","reviews_score"]

for col in int_cols:
    if col in df.columns: df[col] = df[col].astype(int)
for col in float_cols:
    if col in df.columns: df[col] = df[col].astype(float).round(4)

print("✓ Data types verified")
print(df.dtypes[["base_price","month","is_weekend","checkin_date"]])


## Step 6 — Outlier Detection & Treatment

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Before
axes[0].boxplot(df["actual_price"], patch_artist=True,
                boxprops=dict(facecolor="#3b82f6", alpha=0.7))
axes[0].set_title("actual_price — Before Outlier Treatment")
axes[0].set_ylabel("Price ($)")

# IQR capping
price_cols = ["actual_price","optimal_price","base_price",
              "competitor_price_1","competitor_price_2","competitor_price_3","revenue_per_night"]
total_treated = 0
for col in price_cols:
    if col not in df.columns: continue
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 3.0*IQR, Q3 + 3.0*IQR
    mask = (df[col] < lower) | (df[col] > upper)
    n_out = mask.sum()
    if n_out:
        df.loc[df[col] < lower, col] = lower
        df.loc[df[col] > upper, col] = upper
        total_treated += n_out
        print(f"  [{col}] {n_out} outliers capped to [{lower:.1f}, {upper:.1f}]")

# After
axes[1].boxplot(df["actual_price"], patch_artist=True,
                boxprops=dict(facecolor="#10b981", alpha=0.7))
axes[1].set_title("actual_price — After Outlier Treatment")
axes[1].set_ylabel("Price ($)")

plt.suptitle(f"Outlier Treatment — {total_treated} total outliers capped", fontsize=13)
plt.tight_layout(); plt.show()
print(f"\n✓ Total outliers treated: {total_treated}")


## Step 7 — Feature Engineering

In [ ]:
# Competitor price position
df["price_vs_comp_avg"]  = (df["actual_price"] - df["competitor_avg_price"]) / df["competitor_avg_price"]
df["price_vs_comp_min"]  = (df["actual_price"] - df["competitor_min_price"]) / df["competitor_min_price"]
df["price_comp_spread"]  = df["competitor_max_price"] - df["competitor_min_price"]
df["price_premium"]      = df["actual_price"] / df["base_price"]
df["revenue_efficiency"] = df["revpar"] / df["actual_price"]

# Booking window buckets
def lt_bucket(d):
    if d <= 3:  return "Last_Minute"
    if d <= 14: return "Short_Term"
    if d <= 60: return "Medium_Term"
    return "Long_Term"
df["lead_time_bucket"] = df["lead_time_days"].apply(lt_bucket)

# Demand flags
df["high_demand"] = ((df["occupancy_rate"] > 0.80) & (df["demand_score"] > 0.85)).astype(int)
df["has_event"]   = (df["event_type"] != "None").astype(int)
df["price_per_person"] = (df["actual_price"] / 1.8).round(2)

# Cyclical time features
df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
df["dow_sin"]   = np.sin(2 * np.pi * df["day_of_week"] / 7)
df["dow_cos"]   = np.cos(2 * np.pi * df["day_of_week"] / 7)

print("✓ 12 new features engineered:")
new_features = ["price_vs_comp_avg","price_vs_comp_min","price_comp_spread",
                "price_premium","revenue_efficiency","lead_time_bucket",
                "high_demand","has_event","price_per_person",
                "month_sin","month_cos","dow_sin"]
for f in new_features:
    print(f"   + {f}")

# Visualise booking window distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df["lead_time_bucket"].value_counts().plot(kind="bar", ax=axes[0], color="#3b82f6", edgecolor="black")
axes[0].set_title("Booking Window Distribution"); axes[0].set_xlabel("")
df[["price_vs_comp_avg","price_premium"]].hist(bins=30, ax=axes[1] if False else None,
    color=["#f59e0b","#10b981"], alpha=0.7, figsize=(12,4))
pd.DataFrame({"price_premium": df["price_premium"], "price_vs_comp_avg": df["price_vs_comp_avg"]}
).hist(bins=30, color=["#10b981","#f59e0b"], alpha=0.7)
plt.suptitle("Engineered Feature Distributions", fontsize=13)
plt.tight_layout(); plt.show()


## Step 8 — Categorical Encoding (LabelEncoder)

In [ ]:
encoders = {}
le_cols = ["hotel_name","room_type","channel","guest_type",
           "event_type","season","day_of_week_name","lead_time_bucket"]
for col in le_cols:
    if col in df.columns:
        le = LabelEncoder()
        df[f"{col}_enc"] = le.fit_transform(df[col].astype(str))
        encoders[col] = le
        print(f"  [{col}] → {col}_enc  |  {len(le.classes_)} categories: {list(le.classes_[:3])}...")

# Visualize encoding
fig, ax = plt.subplots(figsize=(10, 4))
cat_counts = {col: df[col].nunique() for col in le_cols if col in df.columns}
ax.barh(list(cat_counts.keys()), list(cat_counts.values()), color="#8b5cf6")
ax.set_title("Unique Categories per Encoded Column"); ax.set_xlabel("# of Unique Values")
plt.tight_layout(); plt.show()


## Step 9 — Feature Scaling (StandardScaler)

In [ ]:
FEATURE_COLS = [
    "month","day_of_week","week_of_year","is_weekend","is_holiday",
    "lead_time_days","length_of_stay","base_price",
    "competitor_avg_price","competitor_min_price","competitor_max_price",
    "competitor_price_1","competitor_price_2","competitor_price_3",
    "occupancy_rate","demand_score","weather_score","reviews_score",
    "event_magnitude","repeat_guest","has_event","high_demand",
    "price_vs_comp_avg","price_vs_comp_min","price_comp_spread",
    "price_premium","month_sin","month_cos","dow_sin","dow_cos",
    "hotel_name_enc","room_type_enc","channel_enc",
    "guest_type_enc","event_type_enc","season_enc",
]
existing_feat = [c for c in FEATURE_COLS if c in df.columns]
scaler = StandardScaler()
scaled = scaler.fit_transform(df[existing_feat].fillna(0))
df_scaled = pd.DataFrame(scaled, columns=[f"{c}_scaled" for c in existing_feat], index=df.index)
df = pd.concat([df, df_scaled], axis=1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df["base_price"].hist(bins=30, ax=axes[0], color="#e94560", alpha=0.7)
axes[0].set_title("base_price — Before Scaling")
df["base_price_scaled"].hist(bins=30, ax=axes[1], color="#3b82f6", alpha=0.7)
axes[1].set_title("base_price — After StandardScaler")
plt.tight_layout(); plt.show()
print(f"✓ Scaled {len(existing_feat)} features")


## Step 10 — Save Clean Dataset & Artifacts

In [ ]:
df.to_csv(CLEAN_PATH, index=False)
with open(f"{MODEL_DIR}/encoders.pkl", "wb") as f:
    pickle.dump(encoders, f)
with open(f"{MODEL_DIR}/scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

print("=" * 55)
print("✓ PREPROCESSING COMPLETE")
print("=" * 55)
print(f"  Clean CSV  : {CLEAN_PATH}")
print(f"  Final shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  Encoders   : saved to {MODEL_DIR}/encoders.pkl")
print(f"  Scaler     : saved to {MODEL_DIR}/scaler.pkl")
